# 24_E16 - Integración backend/MVP

Validación del servicio Python generado en E15 como componente integrable al backend/MVP.

Este notebook no entrena modelos. Prueba el paquete `pfi_ai_service`, valida endpoints FastAPI con `TestClient`, genera contratos JSON para backend/frontend y deja un ejemplo de puente Spring Boot → servicio IA.


In [ ]:
# Dependencias mínimas para smoke test de FastAPI
import importlib.util
import sys

packages = {
    "fastapi": "fastapi",
    "httpx": "httpx",
    "pandas": "pandas",
    "pydantic": "pydantic",
    "uvicorn": "uvicorn",
}

missing = [pip_name for module_name, pip_name in packages.items() if importlib.util.find_spec(module_name) is None]
if missing:
    !pip -q install {' '.join(missing)}

print("Dependencias OK")


In [ ]:
from pathlib import Path
import json
import sys
import os
import textwrap
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
REPO_ROOT = PFI_ROOT / "repo"
SERVICE_ROOT = REPO_ROOT / "ai_service"
PACKAGE_ROOT = SERVICE_ROOT / "pfi_ai_service"

E16_ROOT = PFI_ROOT / "results" / "E16_backend_integration_smoke"
DOCS_ROOT = PFI_ROOT / "docs"

for p in [E16_ROOT, DOCS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("PFI_ROOT:", PFI_ROOT)
print("REPO_ROOT:", REPO_ROOT, "| exists:", REPO_ROOT.exists())
print("SERVICE_ROOT:", SERVICE_ROOT, "| exists:", SERVICE_ROOT.exists())
print("PACKAGE_ROOT:", PACKAGE_ROOT, "| exists:", PACKAGE_ROOT.exists())

assert REPO_ROOT.exists(), f"No existe repo: {REPO_ROOT}"
assert PACKAGE_ROOT.exists(), f"No existe paquete: {PACKAGE_ROOT}"


## 1. Patch defensivo de API para JSON limpio

Al convertir DataFrames de pandas a JSON pueden aparecer `NaN`, que FastAPI no siempre serializa correctamente. Esta celda deja `api.py` preparado con un helper `clean_for_json`.


In [ ]:
# Patch defensivo de api.py para respuestas JSON limpias
api_path = PACKAGE_ROOT / "api.py"
api_path.write_text('from __future__ import annotations\n\nimport math\nfrom typing import Any\n\nimport pandas as pd\nfrom fastapi import FastAPI, HTTPException\n\nfrom .settings import get_settings, MODEL_REGISTRY\nfrom .agent import build_agent_decisions, summarize_agent_decisions\nfrom .reporting import build_markdown_summary\n\napp = FastAPI(title="PFI AI Service", version="0.1.0")\n\n\ndef clean_for_json(value: Any) -> Any:\n    """Convierte NaN/inf y objetos numpy/pandas a tipos JSON seguros."""\n    if value is None:\n        return None\n\n    if isinstance(value, float):\n        if math.isnan(value) or math.isinf(value):\n            return None\n        return value\n\n    try:\n        import numpy as np\n        if isinstance(value, np.generic):\n            return clean_for_json(value.item())\n    except Exception:\n        pass\n\n    if isinstance(value, dict):\n        return {str(k): clean_for_json(v) for k, v in value.items()}\n\n    if isinstance(value, list):\n        return [clean_for_json(v) for v in value]\n\n    if isinstance(value, tuple):\n        return [clean_for_json(v) for v in value]\n\n    return value\n\n\n@app.get("/health")\ndef health():\n    settings = get_settings()\n    return clean_for_json({\n        "status": "ok",\n        "pfi_root": str(settings.pfi_root),\n        "human_review_required": True,\n    })\n\n\n@app.get("/models")\ndef models():\n    settings = get_settings()\n    return clean_for_json({\n        "models": MODEL_REGISTRY,\n        "paths": {\n            "sagittal_model_path": str(settings.sagittal_model_path),\n            "axial_model_path": str(settings.axial_model_path),\n        },\n    })\n\n\n@app.get("/agent/worklist")\ndef agent_worklist():\n    settings = get_settings()\n    worklist_path = settings.e14_results_root / "E14_agent_worklist.csv"\n    if not worklist_path.exists():\n        raise HTTPException(status_code=404, detail=f"No existe {worklist_path}")\n\n    df = pd.read_csv(worklist_path)\n    return clean_for_json({\n        "rows": int(len(df)),\n        "items": df.to_dict(orient="records"),\n    })\n\n\n@app.get("/agent/report")\ndef agent_report():\n    settings = get_settings()\n    worklist_path = settings.e14_results_root / "E14_agent_worklist.csv"\n    metrics_path = settings.e14_results_root / "E14_agent_metrics_summary.csv"\n\n    if not worklist_path.exists():\n        raise HTTPException(status_code=404, detail=f"No existe {worklist_path}")\n\n    worklist = pd.read_csv(worklist_path)\n    decisions = build_agent_decisions(worklist)\n\n    if metrics_path.exists():\n        metrics = pd.read_csv(metrics_path)\n        if "agent_item_id" in metrics.columns:\n            merge_cols = [c for c in ["agent_item_id", "plane", "case_ref"] if c in decisions.columns and c in metrics.columns]\n            decisions = decisions.merge(metrics, on=merge_cols, how="left")\n\n    summary = summarize_agent_decisions(decisions)\n\n    return clean_for_json({\n        "summary": summary,\n        "markdown": build_markdown_summary(summary),\n        "items": decisions.to_dict(orient="records"),\n    })\n', encoding="utf-8")
print("API patch escrita:", api_path)


In [ ]:
# Archivos auxiliares mínimos del servicio
readme_path = SERVICE_ROOT / "README.md"
requirements_path = SERVICE_ROOT / "requirements-ai-service.txt"

if not readme_path.exists():
    readme_path.write_text('# PFI AI Service\n\nServicio Python inicial para integrar el agente IA del PFI con backend/MVP.\n\nEndpoints:\n- GET /health\n- GET /models\n- GET /agent/worklist\n- GET /agent/report\n\nLa revisión profesional sigue siendo obligatoria.\n', encoding="utf-8")

if not requirements_path.exists():
    requirements_path.write_text('fastapi>=0.110\nuvicorn[standard]>=0.29\npandas>=2.0\nnumpy>=1.24\npydantic>=2.0\nhttpx>=0.27\n', encoding="utf-8")

print("README exists:", readme_path.exists())
print("requirements exists:", requirements_path.exists())


## 2. Importar paquete y probar endpoints FastAPI con TestClient

In [ ]:
# Importar paquete desde repo/ai_service
if str(SERVICE_ROOT) not in sys.path:
    sys.path.insert(0, str(SERVICE_ROOT))

os.environ["PFI_ROOT"] = str(PFI_ROOT)

import importlib
import pfi_ai_service
from pfi_ai_service.settings import get_settings, MODEL_REGISTRY

import pfi_ai_service.api as service_api
importlib.reload(service_api)

from fastapi.testclient import TestClient

settings = get_settings()
client = TestClient(service_api.app)

print("Package:", pfi_ai_service.__version__)
print("Settings pfi_root:", settings.pfi_root)
print("Models:", list(MODEL_REGISTRY.keys()))
print("E14 worklist:", settings.e14_results_root / "E14_agent_worklist.csv", "->", (settings.e14_results_root / "E14_agent_worklist.csv").exists())
print("E14 metrics:", settings.e14_results_root / "E14_agent_metrics_summary.csv", "->", (settings.e14_results_root / "E14_agent_metrics_summary.csv").exists())


In [ ]:
# Ejecutar smoke test de endpoints
endpoints = [
    "/health",
    "/models",
    "/agent/worklist",
    "/agent/report",
]

endpoint_rows = []
endpoint_responses = {}

for endpoint in endpoints:
    try:
        response = client.get(endpoint)
        status_code = response.status_code
        ok = 200 <= status_code < 300
        data = response.json() if response.content else None

        endpoint_responses[endpoint] = data

        row = {
            "endpoint": endpoint,
            "status_code": status_code,
            "ok": ok,
            "top_level_keys": list(data.keys()) if isinstance(data, dict) else [],
            "error": None,
        }

        if endpoint == "/agent/worklist" and isinstance(data, dict):
            row["rows"] = data.get("rows")
            row["items_len"] = len(data.get("items", []))
        elif endpoint == "/agent/report" and isinstance(data, dict):
            row["items_len"] = len(data.get("items", []))
            row["summary_total_items"] = data.get("summary", {}).get("total_items")
            row["priority_distribution"] = data.get("summary", {}).get("priority_distribution")
        elif endpoint == "/models" and isinstance(data, dict):
            row["model_keys"] = list(data.get("models", {}).keys())

    except Exception as exc:
        row = {
            "endpoint": endpoint,
            "status_code": None,
            "ok": False,
            "top_level_keys": [],
            "error": repr(exc),
        }

    endpoint_rows.append(row)

endpoint_checks_df = pd.DataFrame(endpoint_rows)
endpoint_checks_csv = E16_ROOT / "E16_fastapi_endpoint_checks.csv"
endpoint_checks_df.to_csv(endpoint_checks_csv, index=False)

display(endpoint_checks_df)
print("Endpoint checks:", endpoint_checks_csv)

assert endpoint_checks_df["ok"].all(), "Algún endpoint falló. Revisar endpoint_checks_df."


In [ ]:
# Guardar respuesta sample de /agent/report
agent_report_response = endpoint_responses.get("/agent/report", {})

agent_report_sample_path = E16_ROOT / "E16_agent_report_response_sample.json"
agent_report_sample_path.write_text(
    json.dumps(agent_report_response, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

summary = agent_report_response.get("summary", {})
items = agent_report_response.get("items", [])

print("Agent report sample:", agent_report_sample_path)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("Items:", len(items))


## 3. Validar contrato JSON backend/frontend

In [ ]:
# Validación del contrato mínimo del agente
required_summary_keys = [
    "total_items",
    "plane_distribution",
    "priority_distribution",
    "status_distribution",
]

required_item_keys = [
    "agent_item_id",
    "plane",
    "model_key",
    "case_ref",
    "figure_path",
    "foreground_ratio",
    "n_components",
    "present_classes",
    "flags",
    "mean_confidence",
    "mean_fg_confidence",
    "agent_status",
    "review_priority",
    "agent_reasons",
    "recommended_action",
    "human_review_required",
]

contract_checks = []

for key in required_summary_keys:
    contract_checks.append({
        "scope": "summary",
        "key": key,
        "present": key in summary,
    })

if items:
    first_item = items[0]
    for key in required_item_keys:
        contract_checks.append({
            "scope": "item",
            "key": key,
            "present": key in first_item,
        })

contract_df = pd.DataFrame(contract_checks)
display(contract_df)

assert contract_df["present"].all(), "Faltan claves en el contrato JSON."

contract_summary = {
    "service": "pfi_ai_service",
    "tested_endpoints": endpoints,
    "agent_report_contract": {
        "summary_required_keys": required_summary_keys,
        "item_required_keys": required_item_keys,
    },
    "observed_summary": summary,
    "checks_passed": bool(contract_df["present"].all()),
}

contract_summary_path = E16_ROOT / "E16_backend_contract_summary.json"
contract_summary_path.write_text(
    json.dumps(contract_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Contract summary:", contract_summary_path)


## 4. Documentación para Spring Boot y frontend

In [ ]:
# Escribir documentación backend/frontend/Codex
spring_doc_path = DOCS_ROOT / "E16_spring_boot_bridge_contract.md"
frontend_doc_path = DOCS_ROOT / "E16_frontend_payload_contract.md"
codex_prompt_path = DOCS_ROOT / "E16_codex_backend_prompt.md"

spring_doc_path.write_text('# E16 - Contrato Spring Boot -> Python AI Service\n\n## Servicio Python\n\nBase URL local sugerida:\n\n```text\nhttp://localhost:8000\n```\n\n## Endpoints consumidos\n\n```text\nGET /health\nGET /models\nGET /agent/worklist\nGET /agent/report\n```\n\n## Respuesta principal para backend\n\nEl endpoint `/agent/report` devuelve `summary`, `markdown` e `items`.\n\n## Ejemplo Spring Boot con WebClient\n\n```java\n@Service\npublic class AiServiceClient {\n\n    private final WebClient webClient;\n\n    public AiServiceClient(WebClient.Builder builder) {\n        this.webClient = builder\n            .baseUrl("http://localhost:8000")\n            .build();\n    }\n\n    public Mono<Map> getAgentReport() {\n        return webClient.get()\n            .uri("/agent/report")\n            .retrieve()\n            .bodyToMono(Map.class);\n    }\n\n    public Mono<Map> getHealth() {\n        return webClient.get()\n            .uri("/health")\n            .retrieve()\n            .bodyToMono(Map.class);\n    }\n}\n```\n\n## Recomendación de arquitectura\n\nSpring Boot debería actuar como backend de producto y delegar inferencia/orquestación al servicio Python. El frontend consume Spring Boot, no directamente Python.\n', encoding="utf-8")
frontend_doc_path.write_text('# E16 - Contrato de payload para frontend\n\n## Vista Worklist\n\nCada item de `/agent/report.items[]` puede alimentar una fila de worklist.\n\nCampos principales:\n\n```text\nagent_item_id\nplane\ncase_ref\nmodel_key\nfigure_path\nforeground_ratio\nn_components\npresent_classes\nflags\nmean_confidence\nmean_fg_confidence\nagent_status\nreview_priority\nagent_reasons\nrecommended_action\nhuman_review_required\ndice_macro_useful_classes\niou_macro_useful_classes\n```\n\n## UI sugerida\n\n- Badge por plano: axial/sagital.\n- Badge por prioridad: baja/media/alta.\n- Mostrar overlay usando `figure_path` o una URL servida por backend.\n- Mostrar razones del agente.\n- Mostrar acción recomendada.\n- Confirmar que requiere revisión profesional.\n', encoding="utf-8")
codex_prompt_path.write_text('# Prompt Codex - E16 Backend/MVP bridge\n\nContexto:\nTenemos un servicio Python en `ai_service/pfi_ai_service` con FastAPI. Ya fue validado con TestClient en E16.\n\nObjetivo:\nIntegrar Spring Boot con el servicio Python.\n\nTareas:\n1. Crear configuración `AiServiceProperties` con baseUrl.\n2. Crear cliente `AiServiceClient` usando WebClient.\n3. Crear DTOs para `/agent/report`.\n4. Crear controller Spring Boot `/api/ai/agent/report`.\n5. Manejar errores si Python service no responde.\n6. Mantener human-in-the-loop.\n\nEndpoints Python:\n- GET /health\n- GET /models\n- GET /agent/worklist\n- GET /agent/report\n', encoding="utf-8")

print("Spring contract:", spring_doc_path)
print("Frontend contract:", frontend_doc_path)
print("Codex prompt:", codex_prompt_path)


## 5. Reporte final E16

In [ ]:
# Reporte final E16
report = {
    "notebook": "24_E16_backend_integration_smoke",
    "goal": "validate ai_service as backend/MVP integration component",
    "service_root": str(SERVICE_ROOT),
    "package_root": str(PACKAGE_ROOT),
    "tested_with": "FastAPI TestClient",
    "endpoints": endpoint_checks_df.to_dict(orient="records"),
    "outputs": {
        "endpoint_checks_csv": str(endpoint_checks_csv),
        "agent_report_response_sample_json": str(agent_report_sample_path),
        "backend_contract_summary_json": str(contract_summary_path),
        "spring_boot_bridge_contract_md": str(spring_doc_path),
        "frontend_payload_contract_md": str(frontend_doc_path),
        "codex_backend_prompt_md": str(codex_prompt_path),
    },
    "summary": summary,
    "checks": {
        "package_import_ok": True,
        "fastapi_app_import_ok": True,
        "all_endpoints_ok": bool(endpoint_checks_df["ok"].all()),
        "contract_checks_passed": bool(contract_df["present"].all()),
        "human_review_required": True,
    },
    "decision": "backend_integration_smoke_ready_for_spring_boot_bridge",
}

report_path = E16_ROOT / "E16_backend_integration_report.json"
report_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

conclusion = f'''# E16 - Integración backend/MVP

## Objetivo

Validar el paquete `pfi_ai_service` generado en E15 como componente integrable con backend/MVP.

## Resultado

Se importó correctamente el paquete, se instanció la aplicación FastAPI y se probaron endpoints mediante `TestClient`.

## Endpoints probados

- `/health`
- `/models`
- `/agent/worklist`
- `/agent/report`

## Resumen observado

```json
{json.dumps(summary, indent=2, ensure_ascii=False)}
```

## Decisión

El servicio Python queda listo para ser consumido por un backend Spring Boot mediante HTTP.

## Política del sistema

El servicio conserva revisión humana obligatoria y actúa como apoyo/orquestación, no como diagnóstico automático.

## Próximo paso

Implementar el bridge real Spring Boot -> Python service y luego preparar la visualización frontend de worklist, overlays, prioridades y reportes.
'''

conclusion_path = DOCS_ROOT / "E16_backend_integration_conclusion.md"
conclusion_path.write_text(conclusion, encoding="utf-8")

print("Reporte E16:", report_path)
print("Conclusión E16:", conclusion_path)
print(json.dumps(report, indent=2, ensure_ascii=False))
